## Stage 2 Unlearning - All 4 Methods

Paper: FIUBench (ICLR 2025)  
Goal: Train GA, GD, KL, PO unlearning on Stage 1 checkpoint using exact hyperparameters from Table 7.

| Method | forget_loss | lr   | batch | accum | epochs | split   |
|--------|-------------|------|-------|-------|--------|---------|
| GA     | ga          | 2e-5 | 8     | 16    | 8      | forget5 |
| GD     | gd          | 2e-5 | 8     | 16    | 8      | forget5 |
| KL     | kl          | 1e-4 | 8     | 16    | 8      | forget5 |
| PO     | idk         | 3e-4 | 8     | 16    | 8      | forget5 |

Run cells sequentially. Each method saves its checkpoint to Drive before the next begins.


## Repo + Drive + GPU




In [41]:
import subprocess

result = subprocess.run(
    "git clone https://YOUR_TOKEN@github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

fatal: destination path '/content/FIUBench_Reproducing' already exists and is not an empty directory.



In [42]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

print(f"Pulling latest changes from GitHub...\n")

# Stash local changes first to avoid conflicts
print("Stashing local changes...")
result = subprocess.run(
    "git stash",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)
if result.returncode == 0 and result.stdout.strip():
    print(result.stdout)
else:
    print("(no local changes to stash)")

# Now pull
result = subprocess.run(
    "git pull",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)

if result.returncode == 0:
    print("✅ Repository updated")
    print(result.stdout)
else:
    print("❌ Pull failed")
    print(result.stderr)

Pulling latest changes from GitHub...

Stashing local changes...
No local changes to save

✅ Repository updated
Updating 601c8bd..2c1b6fd
Fast-forward
 FIUBench/config/finetune_llava_phi.yaml |    6 +-
 scripts/week1day4.ipynb                 | 2169 +++++++++++++++++++++++++++++--
 2 files changed, 2043 insertions(+), 132 deletions(-)



## Install dependencies

In [4]:
import subprocess, sys

deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "transformers==4.48.0",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

import transformers
import torch
print(f"✅ torch={torch.__version__}  transformers={transformers.__version__}")


✅ torch=2.4.1+cu121  transformers=4.48.0


In [6]:
import os
from getpass import getpass

# ── OpenAI API Key ────────────────────────────────────────────────────────────
# Required for: GPT-Eval (model utility metric) and APE (adversarial privacy extraction).
# Using getpass so the key is never stored in the notebook or shown in output.
# Run this cell once per session before any eval cells that use GPT-4o-mini.

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('')
    print("✅ OpenAI API key set for this session")
else:
    print("✅ OpenAI API key already set")


✅ OpenAI API key set for this session


## Patches + copy Stage 1 from Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os, re
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'

os.environ['WANDB_MODE']     = 'disabled'
os.environ['HYDRA_FULL_ERROR'] = '1'

# ── 1. Copy Stage 1 from Drive ────────────────────────────────────────────────
if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    print("Copying Stage 1 from Drive...")
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    ret = os.system(f'rsync -ah --progress {STAGE1_DRIVE}/ {STAGE1_LOCAL}/')
    assert ret == 0, "rsync failed"
safetensors = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
assert safetensors, f"No safetensors in {STAGE1_LOCAL}"
print(f"✅ Stage 1 ready: {[f.name for f in safetensors]}")

# ── 2. Patch forget.py ────────────────────────────────────────────────────────
fg_py = Path(FIUBENCH_DIR) / 'forget.py'
src = fg_py.read_text()
patched = src
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
if patched != src:
    fg_py.write_text(patched)
    print("✅ Patched forget.py")
else:
    print("✅ forget.py already patched")

assert 'torch_dtype=torch.bfloat16' in fg_py.read_text(), "bfloat16 patch missing"
assert 'mixed_precision="bf16"'      in fg_py.read_text(), "mixed_precision patch missing"

# ── 3. Patch modeling_llava.py ────────────────────────────────────────────────
import glob
llava_candidates = glob.glob(
    '/usr/local/lib/python*/dist-packages/transformers/models/llava/modeling_llava.py'
)
if llava_candidates:
    llava_path = llava_candidates[0]
    llava_src  = Path(llava_path).read_text()
    llava_patched = re.sub(
        r'n_image_tokens != n_image_features',
        'False',
        llava_src
    )
    if llava_patched != llava_src:
        Path(llava_path).write_text(llava_patched)
        print(f"✅ Patched modeling_llava.py")
    else:
        print("✅ modeling_llava.py already patched")
else:
    print("⚠️  modeling_llava.py not found — may not be needed for this transformers version")

print("\n✅ All patches applied. Ready for Stage 2.")


Copying Stage 1 from Drive...
✅ Stage 1 ready: ['model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors']
✅ Patched forget.py
✅ Patched modeling_llava.py

✅ All patches applied. Ready for Stage 2.


## Stage 2: Gradient Ascent (GA)

In [27]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'ga'
LR           = '2e-5'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29510 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{METHOD}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ GA checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ GA training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ GA failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")

Starting Stage 2 [GA]  lr=2e-5  batch=8  accum=16  epochs=8  split=forget5
2026-04-19 19:26:56.393409: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776626816.415148   10558 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776626816.421837   10558 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776626816.438565   10558 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776626816.438597   10558 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:

## Stage 2: Gradient Difference (GD)

In [28]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'gd'
LR           = '2e-5'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29511 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{METHOD}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ GD checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ GD training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ GD failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


Starting Stage 2 [GD]  lr=2e-5  batch=8  accum=16  epochs=8  split=forget5
2026-04-19 19:32:44.125881: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776627164.147182   13016 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776627164.153670   13016 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776627164.170075   13016 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776627164.170101   13016 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:

## Stage 2: KL Minimization (KL)

In [32]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'kl'
LR           = '1e-4'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29512 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{METHOD}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ KL checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ KL training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ KL failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


Starting Stage 2 [KL]  lr=1e-4  batch=8  accum=16  epochs=8  split=forget5
2026-04-19 19:39:38.368088: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776627578.391650   15923 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776627578.398541   15923 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776627578.415941   15923 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776627578.415985   15923 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:

## Stage 2: Preference Optimization / IDK (PO)

In [33]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'idk'
LABEL        = 'po'
LR           = '3e-4'
STAGE2_LOCAL = f'/content/stage2_{LABEL}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{LABEL}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29513 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{LABEL}_log.txt"
)

print(f"Starting Stage 2 [PO/IDK]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{LABEL}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ PO checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ PO training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ PO failed (exit {ret}). Check /tmp/forget_{LABEL}_log.txt")


Starting Stage 2 [PO/IDK]  lr=3e-4  batch=8  accum=16  epochs=8  split=forget5
2026-04-19 19:45:57.330034: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776627957.353183   18515 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776627957.360590   18515 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776627957.377977   18515 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776627957.378021   18515 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00

## Retain model baseline

In [43]:
# The retain model is fine-tuned on S_R (excludes all forget sets).
# It is the ideal unlearning upper bound used for KS-Test in Day 5.
# Training: same as Stage 1 but with split=retain to exclude forget sets.

import os, subprocess, time
from pathlib import Path

PROJECT_ROOT   = '/content/FIUBench_Reproducing'
FIUBENCH_DIR   = f'{PROJECT_ROOT}/FIUBench'
RETAIN_LOCAL   = '/content/retain_model'
RETAIN_DRIVE   = '/content/drive/MyDrive/fiubench_checkpoints/retain_model'

Path(RETAIN_LOCAL).mkdir(parents=True, exist_ok=True)

# Fine-tune on retain identities using same Stage 1 hyperparameters
# lr=2e-5, batch=8, accum=16, epochs=10, full fine-tuning (no LoRA)
cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29514 finetune.py "
    f"--config-name finetune_llava_phi "
    f"save_dir={RETAIN_LOCAL} "
    f"split=retain "
    f"batch_size=8 "
    f"gradient_accumulation_steps=16 "
    f"2>&1 | tee /tmp/retain_model_log.txt"
)
# Note: model_id, data_path, lr=2e-5, num_epochs=10, seed=0 are yaml defaults — no override needed

print("\nStarting retain model fine-tuning on retain identities...")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(RETAIN_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {RETAIN_LOCAL}/ {RETAIN_DRIVE}/")
    print(f"✅ Retain model saved to {RETAIN_DRIVE}")
else:
    print(f"❌ Retain model training failed. Check /tmp/retain_model_log.txt")


Starting retain model fine-tuning on retain identities...
2026-04-19 20:35:06.146755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776630906.172046   32353 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776630906.179940   32353 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776630906.198453   32353 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776630906.198499   32353 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776630906.19850

## Stage 2 Evaluation - FIUBench Metrics (forget5)

Runs evaluate_util.py on each checkpoint (retain + GA/GD/KL/PO), then aggregates with aggregate_eval_stat.py.

Output per method: Forget Quality (KS-Test p-value), Model Utility (ROUGE/Prob/Truth Ratio harmonic mean).

In [21]:
import os, subprocess, shutil
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'
STAGE1_LOCAL = '/content/stage1_final'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints'
RETAIN_LOCAL = '/content/retain_model'
RETAIN_DRIVE = f'{DRIVE_ROOT}/retain_model'

METHODS = {
    'ga': {'local': '/content/stage2_ga', 'drive': f'{DRIVE_ROOT}/stage2_forget5/ga'},
    'gd': {'local': '/content/stage2_gd', 'drive': f'{DRIVE_ROOT}/stage2_forget5/gd'},
    'kl': {'local': '/content/stage2_kl', 'drive': f'{DRIVE_ROOT}/stage2_forget5/kl'},
    'po': {'local': '/content/stage2_po', 'drive': f'{DRIVE_ROOT}/stage2_forget5/po'},
}

# -- Patch evaluate_util.py ----------------------------------------------------
_eu = Path(f"{FIUBENCH_DIR}/evaluate_util.py")
_eu_src = _eu.read_text()
_eu_new = _eu_src
_eu_new = _eu_new.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
_eu_new = _eu_new.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
_eu_new = _eu_new.replace('model.half().cuda()', 'model.cuda()')
if _eu_new != _eu_src:
    _eu.write_text(_eu_new)
    print('Patched evaluate_util.py: flash_attention_2->sdpa, float16->bfloat16, .half() removed')
else:
    print('evaluate_util.py already patched')

# -- Restore Stage 1 from Drive -------------------------------------------------
if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    print('Restoring stage1 from Drive...')
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah {DRIVE_ROOT}/stage1/ {STAGE1_LOCAL}/")
else:
    print(f'Stage1 at {STAGE1_LOCAL}')

# -- Restore retain model from Drive --------------------------------------------
if not Path(RETAIN_LOCAL).exists() or not list(Path(RETAIN_LOCAL).glob('*.safetensors')):
    print('Restoring retain model from Drive...')
    Path(RETAIN_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah {RETAIN_DRIVE}/ {RETAIN_LOCAL}/")
else:
    print(f'Retain model at {RETAIN_LOCAL}')

# -- Restore Stage 2 checkpoints from Drive -------------------------------------
for method, paths in METHODS.items():
    ckpt = Path(paths['local']) / 'checkpoint.pt'
    if not ckpt.exists():
        print(f"Restoring {method} from Drive...")
        Path(paths['local']).mkdir(parents=True, exist_ok=True)
        os.system(f"rsync -ah {paths['drive']}/ {paths['local']}/")
    else:
        print(f"{method} checkpoint at {ckpt}")

evaluate_util.py already patched
Stage1 at /content/stage1_final
Retain model at /content/retain_model
ga checkpoint at /content/stage2_ga/checkpoint.pt
gd checkpoint at /content/stage2_gd/checkpoint.pt
kl checkpoint at /content/stage2_kl/checkpoint.pt
po checkpoint at /content/stage2_po/checkpoint.pt


In [ ]:
# Evaluate retain model on forget5 + retain5 splits.
# This produces the reference distribution for KS-Test (Forget Quality).
import os, subprocess, time
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
RETAIN_LOCAL = '/content/retain_model'
RETAIN_EVAL  = f'{RETAIN_LOCAL}/eval_results'
Path(RETAIN_EVAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"bash -c 'set -o pipefail; cd {FIUBENCH_DIR} && "
    f"python evaluate_util.py --config-name eval "
    f"model_path={RETAIN_LOCAL} "
    f"save_dir={RETAIN_EVAL} "
    f"'LoRA.r=0' "
    f"'hydra.run.dir=/tmp/hydra_eval_retain' "
    f"2>&1 | tee /tmp/eval_retain_log.txt'"
)

print('Evaluating retain model (forget5 + retain5)...')
print('=' * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {proc.returncode}  |  Time: {h}h {m}m {s}s")

agg = Path(RETAIN_EVAL) / 'retain5_eval_log_aggregated.json'
print(f"{'PASS' if agg.exists() else 'FAIL'} Retain eval log: {agg}")

# --- forget5 full eval (needed for KS-Test) ---
print(f"\nRunning forget5 full-metric eval for {method.upper()}...")
t1 = time.time()
proc2 = subprocess.Popen(
    ['python', 'evaluate_util.py', '--config-name', 'eval',
        f'model_path={STAGE1_LOCAL}',
        'LoRA.r=128', 'LoRA.alpha=256',
        f'LoRA.lora_path={ckpt}',
        f'save_dir={method_dir}/eval_results/',
        'split_list=[forget5]', 'eval_task=[eval_retain_log]',
        'robust_eval=[[rouge]]',
        'batch_size=4', 'perturb_batch_size=4', 'overwrite=true',
        f'hydra.run.dir=/tmp/hydra_{method}_forget5_full'],
    cwd=FIUBENCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc2.stdout:
    print(line, end='', flush=True)
proc2.wait()
elapsed2 = time.time() - t1
h, m, s = int(elapsed2//3600), int((elapsed2%3600)//60), int(elapsed2%60)
print(f"Exit: {proc2.returncode}  |  Time: {h}h {m}m {s}s")
forget_full = Path(method_dir) / 'eval_results' / 'forget5_eval_retain_log.json'
print(f"{'PASS' if forget_full.exists() else 'FAIL'} {method.upper()} forget5 full log: {forget_full}")

Evaluating retain model (forget5 + retain5)...
2026-04-15 16:22:32.148535: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 16:22:32.177338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776270152.200961   23564 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776270152.208449   23564 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776270152.232840   23564 computation_placer.cc:177] computation placer already regi

In [22]:
# Evaluate each unlearning method on forget5 + retain5.
# Restores Stage 2 checkpoints from Drive at the start — safe to re-run after runtime restart.
import os, subprocess, time
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
STAGE1_LOCAL = '/content/stage1_final'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints'

METHODS = {
    'ga': {'local': '/content/stage2_ga', 'drive': f'{DRIVE_ROOT}/stage2_forget5/ga'},
    'gd': {'local': '/content/stage2_gd', 'drive': f'{DRIVE_ROOT}/stage2_forget5/gd'},
    'kl': {'local': '/content/stage2_kl', 'drive': f'{DRIVE_ROOT}/stage2_forget5/kl'},
    'po': {'local': '/content/stage2_po', 'drive': f'{DRIVE_ROOT}/stage2_forget5/po'},
}

# -- Restore Stage 1 if missing -----------------------------------------------
if not list(Path(STAGE1_LOCAL).glob('*.safetensors') if Path(STAGE1_LOCAL).exists() else []):
    print('Restoring Stage 1 from Drive...')
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    ret = os.system(f'rsync -ah {DRIVE_ROOT}/stage1/ {STAGE1_LOCAL}/')
    assert ret == 0, 'rsync stage1 failed'
print(f'Stage 1 ready: {STAGE1_LOCAL}')

# -- Restore Stage 2 checkpoints from Drive -----------------------------------
available = []
for method, paths in METHODS.items():
    ckpt = Path(paths['local']) / 'checkpoint.pt'
    if not ckpt.exists():
        drive_src = paths['drive']
        if Path(drive_src).exists():
            print(f'Restoring {method} from Drive...')
            Path(paths['local']).mkdir(parents=True, exist_ok=True)
            ret = os.system(f"rsync -ah {drive_src}/ {paths['local']}/")
            if ret == 0 and ckpt.exists():
                print(f'  ✅ {method}: checkpoint.pt restored')
                available.append(method)
            else:
                print(f'  ❌ {method}: rsync returned {ret} or checkpoint.pt missing after sync')
        else:
            print(f'  ⚠️  {method}: not on Drive at {drive_src} — skipping')
    else:
        print(f'  ✅ {method}: checkpoint.pt already present')
        available.append(method)

print(f'
Methods to evaluate: {available}')

# -- Run evaluation for each available method ---------------------------------
for method in available:
    method_dir = METHODS[method]['local']
    ckpt       = f"{method_dir}/checkpoint.pt"
    save_dir   = f"{method_dir}/eval_results/"
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    # --- retain5 + forget5 standard eval (EM, ROUGE, GPT, Truth Ratio) -------
    print(f"
{'='*70}")
    print(f"Evaluating {method.upper()} — retain5 + forget5 standard eval...")
    print(f"{'='*70}")
    t0 = time.time()
    proc = subprocess.Popen(
        ['python', 'evaluate_util.py', '--config-name', 'eval',
         f'model_path={STAGE1_LOCAL}',
         'LoRA.r=128', 'LoRA.alpha=256', f'LoRA.lora_path={ckpt}',
         f'save_dir={save_dir}',
         f'hydra.run.dir=/tmp/hydra_eval_{method}'],
        cwd=FIUBENCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    elapsed = time.time() - t0
    h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    print(f"
Exit: {proc.returncode}  |  Time: {h}h {m}m {s}s")
    agg = Path(save_dir) / 'retain5_eval_log_aggregated.json'
    print(f"{'PASS' if agg.exists() else 'FAIL'} Retain5 eval log: {agg}")

    # --- forget5 full-metric eval (MIA, APE, KS-Test inputs) -----------------
    print(f"
Running forget5 full-metric eval for {method.upper()}...")
    t1 = time.time()
    proc2 = subprocess.Popen(
        ['python', 'evaluate_util.py', '--config-name', 'eval',
         f'model_path={STAGE1_LOCAL}',
         'LoRA.r=128', 'LoRA.alpha=256', f'LoRA.lora_path={ckpt}',
         f'save_dir={save_dir}',
         'split_list=[forget5]', 'eval_task=[eval_retain_log]',
         'robust_eval=[[rouge]]',
         'batch_size=4', 'perturb_batch_size=4', 'overwrite=true',
         f'hydra.run.dir=/tmp/hydra_{method}_forget5_full'],
        cwd=FIUBENCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc2.stdout:
        print(line, end='', flush=True)
    proc2.wait()
    elapsed2 = time.time() - t1
    h, m, s = int(elapsed2//3600), int((elapsed2%3600)//60), int(elapsed2%60)
    print(f"Exit: {proc2.returncode}  |  Time: {h}h {m}m {s}s")
    forget_full = Path(save_dir) / 'forget5_eval_retain_log.json'
    print(f"{'PASS' if forget_full.exists() else 'FAIL'} Forget5 full log: {forget_full}")


Stage 1 ready: /content/stage1_final
  ✅ ga: checkpoint.pt already present
  ✅ gd: checkpoint.pt already present
  ✅ kl: checkpoint.pt already present
  ✅ po: checkpoint.pt already present

Methods to evaluate: ['ga', 'gd', 'kl', 'po']

Evaluating GA (forget5 + retain5)...
2026-04-15 17:25:13.978594: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 17:25:14.006060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776273914.028693   39419 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776273914.036453   39419 cuda_blas

## Save results to drive

In [26]:
import subprocess, os

DRIVE_STAGE2 = "/content/drive/MyDrive/fiubench_checkpoints/stage2"
LOCAL_STAGE2 = "/content"

for method in ["ga", "gd", "kl", "po"]:
    src = f"{LOCAL_STAGE2}/stage2_{method}/eval_results/"
    dst = f"{DRIVE_STAGE2}/stage2_{method}/eval_results/"
    os.makedirs(dst, exist_ok=True)
    ret = subprocess.call(["rsync", "-av", src, dst])
    print(f"{method}: rsync returned {ret}")


ga: rsync returned 0
gd: rsync returned 0
kl: rsync returned 0
po: rsync returned 0


## Calculate Results

In [ ]:
# Aggregate results: compute Forget Quality (KS-Test) + Model Utility for each method.
import os, json, csv, subprocess
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
RESULTS_DIR  = '/content/eval_results'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

RETAIN_DIR = '/content/retain_model/eval_results'
METHODS = {
    'GA': '/content/stage2_ga/eval_results',
    'GD': '/content/stage2_gd/eval_results',
    'KL': '/content/stage2_kl/eval_results',
    'PO': '/content/stage2_po/eval_results',
}

def build_combined_json(eval_dir, tmp_name):
    """Combine forget5_eval_retain_log + retain5_eval_retain_log into the
    format aggregate_eval_stat.py expects."""
    forget_path = Path(eval_dir) / 'forget5_eval_retain_log.json'
    retain_path = Path(eval_dir) / 'retain5_eval_retain_log.json'
    if not forget_path.exists():
        print(f"  MISSING: {forget_path}")
        return None
    if not retain_path.exists():
        print(f"  MISSING: {retain_path}")
        return None
    combined = {
        'eval_forget_log.json': json.load(open(forget_path)),
        'eval_retain_log.json': json.load(open(retain_path)),
    }
    tmp_path = f'/tmp/{tmp_name}_combined.json'
    with open(tmp_path, 'w') as f:
        json.dump(combined, f)
    return tmp_path

# Build retain reference combined JSON
retain_combined = build_combined_json(RETAIN_DIR, 'retain')
if not retain_combined:
    print("Retain eval incomplete — run cells 23 first.")
else:
    all_results = []
    for method, eval_dir in METHODS.items():
        ckpt_combined = build_combined_json(eval_dir, method.lower())
        if not ckpt_combined:
            print(f"Skipping {method}: forget5 full eval not yet run.")
            continue

        save_csv = f"{RESULTS_DIR}/{method.lower()}_aggr.csv"
        result = subprocess.run(
            ['python', 'aggregate_eval_stat.py',
             f'retain_result={retain_combined}',
             f'ckpt_result={ckpt_combined}',
             f'method_name={method}',
             f'save_file={save_csv}',
             f'hydra.run.dir=/tmp/hydra_agg_{method.lower()}'],
            cwd=FIUBENCH_DIR, capture_output=True, text=True
        )
        if result.returncode == 0 and Path(save_csv).exists():
            rows = list(csv.DictReader(open(save_csv)))
            if rows:
                all_results.append(rows[0])
                print(f"✅ {method}: aggregated")
        else:
            print(f"❌ {method}: aggregate_eval_stat.py failed")
            print(result.stderr[-500:])

    if all_results:
        print('\n' + '='*90)
        print('FIUBench Reproduced Results (forget5)')
        print('='*90)
        keys = ['Method', 'Forget Quality', 'Model Utility',
                'ROUGE Forget', 'ROUGE Retain', 'Prob. Forget', 'Prob. Retain',
                'Truth Ratio Forget', 'Truth Ratio Retain', 'GPT Retain', 'EM Retain']
        header = f"{'Method':<8}" + ''.join(f"  {k:<20}" for k in keys[1:])
        print(header)
        print('-' * len(header))
        for r in all_results:
            row = f"{r.get('Method',''):<8}"
            for k in keys[1:]:
                v = r.get(k, 'N/A')
                try:    row += f"  {float(v):<20.4f}"
                except: row += f"  {str(v):<20}"
            print(row)
        print('='*90)

        DRIVE_RESULTS = '/content/drive/MyDrive/fiubench_checkpoints/eval_results_forget5'
        os.system(f"rsync -ah {RESULTS_DIR}/ {DRIVE_RESULTS}/")
        print(f"\nResults synced to Drive: {DRIVE_RESULTS}")
    else:
        print("\nNo results — run forget5 full eval in cells 23 and 24 first.")

  MISSING: /content/retain_model/eval_results/forget5_eval_retain_log.json
Retain eval incomplete — run cells 23 first.


## Verify Claims

In [ ]:
# ── Phase 2 Analysis: Verify All Claims
import pandas as pd
import json
from pathlib import Path

results_dir = '/content/eval_results'
results = {}

# Load aggregated results
for method in ['ga', 'gd', 'kl', 'po']:
    csv_path = Path(results_dir) / f'{method}_aggr.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        results[method.upper()] = {
            'Forget Quality': float(df['Forget Quality'].iloc[0]),
            'Model Utility': float(df['Model Utility'].iloc[0])
        }

print("=" * 90)
print("FIUBENCH REPRODUCTION ANALYSIS - PHASE 2")
print("=" * 90)

# ── CLAIM 1: Core Tradeoff
print("\n[CLAIM 1] Core Tradeoff: GA/GD > KL/PO (Forget); GA/GD < KL/PO (Utility)")
print("-" * 90)
for method in ['GA', 'GD', 'KL', 'PO']:
    if method in results:
        fq = results[method]['Forget Quality']
        mu = results[method]['Model Utility']
        print(f"  {method}: Forget Quality={fq:.4f}  Model Utility={mu:.4f}")

fq_ga_gd = [results[m]['Forget Quality'] for m in ['GA', 'GD'] if m in results]
fq_kl_po = [results[m]['Forget Quality'] for m in ['KL', 'PO'] if m in results]
mu_ga_gd = [results[m]['Model Utility'] for m in ['GA', 'GD'] if m in results]
mu_kl_po = [results[m]['Model Utility'] for m in ['KL', 'PO'] if m in results]

fq_claim = sum(fq_ga_gd)/len(fq_ga_gd) > sum(fq_kl_po)/len(fq_kl_po) if fq_ga_gd and fq_kl_po else None
mu_claim = sum(mu_ga_gd)/len(mu_ga_gd) < sum(mu_kl_po)/len(mu_kl_po) if mu_ga_gd and mu_kl_po else None

print(f"\n  ✓ GA/GD Forget (avg): {sum(fq_ga_gd)/len(fq_ga_gd):.4f} > KL/PO Forget (avg): {sum(fq_kl_po)/len(fq_kl_po):.4f}? {fq_claim}")
print(f"  ✓ GA/GD Utility (avg): {sum(mu_ga_gd)/len(mu_ga_gd):.4f} < KL/PO Utility (avg): {sum(mu_kl_po)/len(mu_kl_po):.4f}? {mu_claim}")
print(f"  → CLAIM 1 VERIFIED: {fq_claim and mu_claim}" if fq_claim is not None and mu_claim is not None else "  → INCOMPLETE: Need all 4 methods")

# ── CLAIM 2: Step Sensitivity
print("\n[CLAIM 2] Step Sensitivity: GA/GD/KL degrade with training steps")
print("-" * 90)
print("  Status: Step-level checkpoints saved (every 6 steps) up to step_60")
print("  Analysis: Requires per-checkpoint metric extraction")
print("  → TODO: Load eval results from each step folder and plot degradation curves")

# ── CLAIM 3: PO Plateau
print("\n[CLAIM 3] PO Plateau: Forget Quality saturates regardless of steps")
print("-" * 90)
po_fq = results.get('PO', {}).get('Forget Quality', None)
if po_fq:
    print(f"  PO Final Forget Quality: {po_fq:.4f}")
    print("  → Check if PO early (step_6) ≈ PO final (step_60)")
else:
    print("  → INCOMPLETE: PO results needed")

# ── Summary
print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)
print(f"  Methods trained: {len(results)}/4")
print(f"  Claim 1 (Tradeoff): {'✓ VERIFIED' if (fq_claim and mu_claim) else '⚠️  PENDING' if (fq_claim is None or mu_claim is None) else '✗ FAILED'}")
print(f"  Claim 2 (Step Sensitivity): ⚠️  REQUIRES STEP-LEVEL ANALYSIS")
print(f"  Claim 3 (PO Plateau): {'⚠️  REQUIRES CHECKPOINT COMPARISON' if po_fq else '✗ MISSING'}")
print("=" * 90)
